In [292]:
import NS as ns
import pandas as pd
import os
import mysql.connector as SQLconnect
from datetime import datetime as dt 
import numpy as np
import math


In [11]:
#transform weatherdata into database




succes


In [75]:
#step one: get data out of CSV file
        
files = {dt.strptime(file[26:34], '%Y%m%d') : 'weather_data/' + file for file in os.listdir('weather_data') if '.csv' in file} #bitchin' dict comprehension which gets the csv file out of the directory and links it to a date.

for date in files:
    print(f'{date}: {files[date]}')

raw_weather_data = {date: pd.read_csv(files[date]) for date in files}


2026-01-25 00:00:00: weather_data/KMDS__OPER_P___10M_OBS_L2_202601252130.nc.csv
2026-01-27 00:00:00: weather_data/KMDS__OPER_P___10M_OBS_L2_202601271700.nc.csv
2026-01-28 00:00:00: weather_data/KMDS__OPER_P___10M_OBS_L2_202601281130.nc.csv


In [133]:
#step two: put datatypes into weathertypes database
db = ns.openconnection()

cursor = db.cursor()

val = [
    ('DP',),
    ('DM',), 
    ('DT',), 
    ('MP',), 
    ('MM',), 
    ('MT',),
    ('Null',)
]

sql = 'INSERT INTO weertype VALUES (%s)'

cursor.executemany(sql, val)

db.close()

Opening connection to database...
Connection established!


IntegrityError: 1062 (23000): Duplicate entry 'DP' for key 'weertype.PRIMARY'

In [300]:
#step three: write function to translate weatherdata into usable type data
    #take into account missing data in a measurement

def get_classification(file):
    #step 1: open seedfile and extract values for 6 classes
    seed_data = pd.read_csv(file)

    seed_values = {}

    for n in ['DP', 'DM', 'DT', 'MP', 'MM', 'MT']:
        seed_values[n] = {
        'temp': seed_data[f'{n}_temp'].mean() + 10.5,
        'dewp': seed_data[f'{n}_dewp'].mean() + 7
    }

    return(seed_values)


def calculate_dewpoint(temp, rh):
    #Calculate the dewpoint based on magnus formula
    b = 17.625
    c = 243.04
    
    # Calculate gamma
    gamma = np.log(rh / 100.0) + (b * temp) / (c + temp)
    
    # Calculate the dewpoint
    dewpoint = (c * gamma) / (b - gamma)
    return dewpoint



def classify_weather(data, seed):
    #step 1: determine if the data is usable: if not, return 'Null' 
    if pd.isna(data.tg) or pd.isna(data.rh):
        return None
    data = [
        data.tg, calculate_dewpoint(data.tg ,data.rh)
    ]

    #calculate distance to class and pick closest
    closest = [9999, '']
    for classification in seed:
        dis = math.dist(data, [seed[classification]['temp'], seed[classification]['dewp']])
        if dis < closest[0]:
            closest = [dis, classification]
    return(closest[1])
    
    
    

In [393]:
#step four execute function for all available weatherdata and put it into the database
db = ns.openconnection()
cursor = db.cursor()

seed = get_classification('dutch_seeds.csv')

for date in raw_weather_data:
    #step 1: check if data for the day is already present

    sql = 'SELECT * FROM weer WHERE tijd = %s'
    val = [date]

    cursor.execute(sql, val)

    if cursor.fetchone() != None:
        print('datapresent')
        cursor.fetchall()
        continue
    

    #step 2: for each record, transform data into a usable record with type, lon, lat and insert it into the database
    for row in test_data.itertuples():
        cl = classify_weather(row, seed) 
        if cl == None:
            cursor.fetchall()
            continue
        cursor.fetchall()
        #check if the location is already present, and if not, insert it

        sql = 'SELECT * FROM location WHERE lat = %s AND lon = %s'
        val = [row.lat, row.lon]

        cursor.execute(sql, val)
        
        if cursor.fetchone() == None:
            cursor.fetchall()
            sql = 'INSERT INTO location (naam, lat, lon) VALUES (%s, %s, %s)'
            val = [row.stationname, row.lat, row.lon]

            cursor.execute(sql, val)
        cursor.fetchall()

        #find location id for station
        cursor.execute('SELECT id FROM location WHERE (lat, lon) = (%s, %s)', [row.lat, row.lon])

        loc_id = cursor.fetchall()[0][0]

        #get type
        type = classify_weather(row, seed)
        

        #insert
        val = [loc_id, weather_type, date]
        sql = 'INSERT INTO weer (locatie, `type`, tijd) VALUES (%s, %s, %s)'

        cursor.execute(sql, val)
        
        db.commit()
        
      



cursor.execute('SELECT * FROM weer')

db.close()  

Opening connection to database...
Connection established!
datapresent
datapresent
datapresent


In [ ]:
#step seven: use the dataframe to create a table

In [304]:
#weatherdata test
key = []
for x in raw_weather_data:
    key.append(x)

key = key[1]

test_data = raw_weather_data[key]

seed = get_classification('dutch_seeds.csv')

for row in test_data.itertuples():
    print(classify_weather(row, seed))


None
None
None
None
None
None
None
None
DP
None
None
None
None
None
DP
None
None
None
None
DP
DP
None
DP
DP
DP
None
DP
DP
DP
DP
DP
DP
DP
DP
DP
DP
None
DP
DP
DP
DP
None
DP
DP
DP
DP
DP
DP
DP
MM
MM
MT
